In [1]:
# ======================================================
# CELL 2: INITIALIZE SPARK SESSION
# ======================================================

import os
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("CSCI316-Boosting")
    .master("local[2]")                 # CPU cores
    .config("spark.driver.memory", "4g") # Driver RAM
    .config("spark.executor.memory", "4g") # Executor RAM
    .config("spark.sql.shuffle.partitions", "50")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("ERROR")
spark.sparkContext.setCheckpointDir("/app/checkpoints")

print("Spark Version:", spark.version)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/02/12 20:41:26 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/02/12 20:41:28 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/02/12 20:41:28 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.
26/02/12 20:41:28 WARN Utils: Service 'SparkUI' could not bind on port 4042. Attempting port 4043.
26/02/12 20:41:28 WARN Utils: Service 'SparkUI' could not bind on port 4043. Attempting port 4044.


Spark Version: 4.1.1


In [2]:
# ======================================================
# CELL 3: EXTRACT PARQUET FILES
# ======================================================

import os
import shutil

BASE_DIR = "/app"
DATA_PROCESSED = os.path.join(BASE_DIR, "data", "processed")
OUTPUTS_DIR = os.path.join(BASE_DIR, "outputs")

os.makedirs(DATA_PROCESSED, exist_ok=True)

# If the zip exists (legacy from Colab), unpack it into data/processed
zip_candidates = [
    os.path.join(DATA_PROCESSED, "valuation_ml_cv_ready.zip"),
    os.path.join(OUTPUTS_DIR, "artifacts", "valuation_ml_cv_ready.zip"),
    os.path.join(BASE_DIR, "valuation_ml_cv_ready.zip"),
]

zip_path = next((p for p in zip_candidates if os.path.exists(p)), None)

if zip_path:
    print("Found zip:", zip_path)
    shutil.unpack_archive(zip_path, DATA_PROCESSED)
    print("Unzipped into:", DATA_PROCESSED)
else:
    print("No zip found (expected if parquet folder already exists).")

Found zip: /app/outputs/artifacts/valuation_ml_cv_ready.zip
Unzipped into: /app/data/processed


In [3]:
# ======================================================
# CELL 4: LOAD AND CACHE DATASET
# ======================================================

import os
from pyspark.sql import functions as F

BASE_DIR = "/app"
DATA_PROCESSED = os.path.join(BASE_DIR, "data", "processed")

path = os.path.join(DATA_PROCESSED, "valuation_ml_cv_ready.parquet")

if not os.path.exists(path):
    raise FileNotFoundError(
        f"Could not find: {path}\n"
        "Run 03_cross_validation.ipynb first to generate valuation_ml_cv_ready.parquet."
    )

df = spark.read.parquet(path)

df = df.cache()
df.count()   # force cache

print("Rows:", df.count())
print("Columns:", df.columns)

df.printSchema()
df.show(5, truncate=False)

Rows: 88092
Columns: ['row_id', 'features', 'label', 'fold_id']
root
 |-- row_id: integer (nullable = true)
 |-- features: vector (nullable = true)
 |-- label: double (nullable = true)
 |-- fold_id: integer (nullable = true)



+------+---------------------------------------------------------------+-----+-------+
|row_id|features                                                       |label|fold_id|
+------+---------------------------------------------------------------+-----+-------+
|1     |(291,[0,1,2,5,13,174],[2012.0,172615.17,172615.17,1.0,1.0,1.0])|2.0  |1      |
|2     |(291,[0,1,2,6,11,73],[2012.0,145.86,145.86,1.0,1.0,1.0])       |0.0  |2      |
|3     |(291,[0,1,2,5,12,176],[2017.0,2557.34,2557.34,1.0,1.0,1.0])    |2.0  |3      |
|4     |(291,[0,1,2,5,9,112],[2003.0,1111.86,1111.83,1.0,1.0,1.0])     |2.0  |4      |
|5     |(291,[0,1,2,5,10,109],[2010.0,7569.55,13604.62,1.0,1.0,1.0])   |1.0  |5      |
+------+---------------------------------------------------------------+-----+-------+
only showing top 5 rows


In [4]:
# ======================================================
# CELL 5: DETECT LABEL / FEATURES / FOLD COLUMNS
# ======================================================

cols = set(df.columns)

label_candidates = ["label", "Label", "target", "y"]
features_candidates = ["features", "Features", "feature_vector"]
fold_candidates = ["fold_id", "fold", "kfold", "cv_fold"]

def pick_first(existing, candidates):
    for c in candidates:
        if c in existing:
            return c
    return None

LABEL_COL = pick_first(cols, label_candidates)
FEATURES_COL = pick_first(cols, features_candidates)
FOLD_COL = pick_first(cols, fold_candidates)

print("LABEL:", LABEL_COL)
print("FEATURES:", FEATURES_COL)
print("FOLD:", FOLD_COL)

if LABEL_COL is None or FEATURES_COL is None or FOLD_COL is None:
    raise ValueError("Missing required columns")

LABEL: label
FEATURES: features
FOLD: fold_id


In [5]:
# ======================================================
# CELL 6: CHECK NUMBER OF CLASSES
# ======================================================

label_counts = df.groupBy(LABEL_COL).count().orderBy(LABEL_COL)
label_counts.show()

K = label_counts.count()
print("Number of Classes:", K)

+-----+-----+
|label|count|
+-----+-----+
|  0.0|29332|
|  1.0|29324|
|  2.0|29436|
+-----+-----+

Number of Classes: 3


In [6]:
# ======================================================
# CELL 7: METRIC FUNCTIONS
# ======================================================

from pyspark.sql import Window

def compute_metrics(pred_df, label_col, pred_col):

    total = pred_df.count()
    correct = pred_df.filter(F.col(label_col) == F.col(pred_col)).count()

    accuracy = correct / total if total > 0 else 0.0

    tp = (
        pred_df
        .filter(F.col(label_col) == F.col(pred_col))
        .groupBy(label_col)
        .count()
        .withColumnRenamed(label_col, "cls")
        .withColumnRenamed("count", "tp")
    )

    pred_counts = (
        pred_df
        .groupBy(pred_col)
        .count()
        .withColumnRenamed(pred_col, "cls")
        .withColumnRenamed("count", "pred_total")
    )

    actual_counts = (
        pred_df
        .groupBy(label_col)
        .count()
        .withColumnRenamed(label_col, "cls")
        .withColumnRenamed("count", "actual_total")
    )

    per = (
        actual_counts
        .join(pred_counts, "cls", "full")
        .join(tp, "cls", "left")
        .na.fill(0)
    )

    per = (
        per
        .withColumn("fp", F.col("pred_total") - F.col("tp"))
        .withColumn("fn", F.col("actual_total") - F.col("tp"))
    )

    per = (
        per
        .withColumn(
            "precision",
            F.when(F.col("pred_total") == 0, 0.0)
             .otherwise(F.col("tp") / F.col("pred_total"))
        )
        .withColumn(
            "recall",
            F.when(F.col("actual_total") == 0, 0.0)
             .otherwise(F.col("tp") / F.col("actual_total"))
        )
    )

    per = per.withColumn(
        "f1",
        F.when(
            (F.col("precision") + F.col("recall")) == 0,
            0.0
        ).otherwise(
            2 * F.col("precision") * F.col("recall") /
            (F.col("precision") + F.col("recall"))
        )
    )

    rows = per.collect()

    return {
        "accuracy": accuracy,
        "precision": sum(r["precision"] for r in rows) / len(rows),
        "recall": sum(r["recall"] for r in rows) / len(rows),
        "f1": sum(r["f1"] for r in rows) / len(rows)
    }

In [7]:
# ======================================================
# CELL 8: BOOSTING MODEL (SAMME)
# ======================================================

from pyspark.ml.classification import DecisionTreeClassifier
import math


def adaboost_train_predict(train_df, test_df,
                           features_col, label_col,
                           T=10, max_depth=1, seed=42):

    train = train_df.withColumn("row_id", F.monotonically_increasing_id())
    test = test_df.withColumn("row_id", F.monotonically_increasing_id())

    n_train = train_df.count()

    train = train.withColumn("w", F.lit(1.0 / n_train))

    K = train.select(label_col).distinct().count()

    votes = None

    for t in range(T):

        dt = DecisionTreeClassifier(
            featuresCol=features_col,
            labelCol=label_col,
            predictionCol="pred",
            maxDepth=max_depth,
            seed=seed + t,
            weightCol="w"
        )

        model = dt.fit(train)

        train_pred = model.transform(train)\
                          .select("row_id", label_col, "pred", "w")

        # Weighted error
        err_row = (
            train_pred
            .withColumn(
                "wbad",
                F.col("w") *
                (F.col(label_col) != F.col("pred")).cast("double")
            )
            .agg(F.sum("wbad").alias("e"), F.sum("w").alias("s"))
            .collect()[0]
        )

        err = err_row["e"] / err_row["s"]
        err = min(max(err, 1e-12), 1 - 1e-12)

        alpha = math.log((1 - err) / err) + math.log(K - 1)

        # Update weights (INLINE, no bad column)
        upd = train_pred.withColumn(
            "w_new",
            F.col("w") * F.exp(
                F.lit(alpha) *
                (F.col(label_col) != F.col("pred")).cast("double")
            )
        )

        wsum = upd.agg(F.sum("w_new")).collect()[0][0]

        train = (
            train.drop("w")
            .join(upd.select("row_id", "w_new"), "row_id")
            .withColumn("w", F.col("w_new") / F.lit(wsum))
            .drop("w_new")
            .checkpoint(eager=True)
        )

        test_pred = model.transform(test)\
                         .select(
                             "row_id",
                             F.col("pred").cast("int").alias("pred_class")
                         )

        vote = test_pred.withColumn("alpha", F.lit(alpha))

        votes = vote if votes is None else votes.unionByName(vote)

    score = (
        votes
        .groupBy("row_id", "pred_class")
        .agg(F.sum("alpha").alias("score"))
    )

    w = Window.partitionBy("row_id").orderBy(F.col("score").desc())

    final_pred = (
        score
        .withColumn("rn", F.row_number().over(w))
        .filter(F.col("rn") == 1)
        .select("row_id", F.col("pred_class").alias("prediction"))
    )

    return test.select("row_id", label_col)\
               .join(final_pred, "row_id")

In [8]:
# ======================================================
# CELL 9: MANUAL 10-FOLD CROSS VALIDATION
# ======================================================

import pandas as pd
import time

folds = [r[FOLD_COL] for r in df.select(FOLD_COL).distinct().orderBy(FOLD_COL).collect()]

results = []

T_ROUNDS = 10
MAX_DEPTH = 1
SEED = 42

total_folds = len(folds)
start_time = time.time()

print("Total folds:", total_folds)
print("Boosting rounds per fold:", T_ROUNDS)
print("=" * 50)


for i, k in enumerate(folds, start=1):

    fold_start = time.time()

    print(f"[{i}/{total_folds}] Starting Fold {k} "
          f"({round(i/total_folds*100,2)}%)")

    train_df = df.filter(F.col(FOLD_COL) != k).select(FEATURES_COL, LABEL_COL)
    test_df = df.filter(F.col(FOLD_COL) == k).select(FEATURES_COL, LABEL_COL)

    pred = adaboost_train_predict(
        train_df,
        test_df,
        FEATURES_COL,
        LABEL_COL,
        T=T_ROUNDS,
        max_depth=MAX_DEPTH,
        seed=SEED
    )

    m = compute_metrics(pred, LABEL_COL, "prediction")
    m["fold"] = k

    results.append(m)

    fold_time = time.time() - fold_start
    elapsed = time.time() - start_time

    print(f"[{i}/{total_folds}] Finished Fold {k}")
    print("   Metrics:", m)
    print(f"   Fold time: {round(fold_time,2)} secs")
    print(f"   Elapsed:   {round(elapsed,2)} secs")
    print("-" * 50)


results_df = pd.DataFrame(results)

print("All folds completed.")
print("Total time:", round((time.time() - start_time),2), "seconds")

results_df

Total folds: 10
Boosting rounds per fold: 10
[1/10] Starting Fold 0 (10.0%)


[1/10] Finished Fold 0
   Metrics: {'accuracy': 0.7200590305369509, 'precision': 0.7276021030922716, 'recall': 0.7195048323789578, 'f1': 0.7219728603365136, 'fold': 0}
   Fold time: 88.33 secs
   Elapsed:   88.34 secs
--------------------------------------------------
[2/10] Starting Fold 1 (20.0%)


[2/10] Finished Fold 1
   Metrics: {'accuracy': 0.7314415437003405, 'precision': 0.7385256007258861, 'recall': 0.7317545334242316, 'f1': 0.7344279759167697, 'fold': 1}
   Fold time: 47.16 secs
   Elapsed:   135.49 secs
--------------------------------------------------
[3/10] Starting Fold 2 (30.0%)


[3/10] Finished Fold 2
   Metrics: {'accuracy': 0.7174801362088535, 'precision': 0.742330758167809, 'recall': 0.7178382857227201, 'f1': 0.7234083395493851, 'fold': 2}
   Fold time: 42.47 secs
   Elapsed:   177.96 secs
--------------------------------------------------
[4/10] Starting Fold 3 (40.0%)


[4/10] Finished Fold 3
   Metrics: {'accuracy': 0.721307753433988, 'precision': 0.7284754038016934, 'recall': 0.7211185341530616, 'f1': 0.7233388879884242, 'fold': 3}
   Fold time: 41.68 secs
   Elapsed:   219.64 secs
--------------------------------------------------
[5/10] Starting Fold 4 (50.0%)


...

[7/10] Finished Fold 6
   Metrics: {'accuracy': 0.7201725508003178, 'precision': 0.7228540092113577, 'recall': 0.7210856121716845, 'f1': 0.7216071784425558, 'fold': 6}
   Fold time: 42.43 secs
   Elapsed:   346.23 secs
--------------------------------------------------
[8/10] Starting Fold 7 (80.0%)


[8/10] Finished Fold 7
   Metrics: {'accuracy': 0.7184697468498127, 'precision': 0.722211051063382, 'recall': 0.7186585551496062, 'f1': 0.7196915731651995, 'fold': 7}
   Fold time: 41.36 secs
   Elapsed:   387.59 secs
--------------------------------------------------
[9/10] Starting Fold 8 (90.0%)


[9/10] Finished Fold 8
   Metrics: {'accuracy': 0.7276648881825406, 'precision': 0.7354054136469133, 'recall': 0.7275766619951617, 'f1': 0.7305559094301501, 'fold': 8}
   Fold time: 54.59 secs
   Elapsed:   442.18 secs
--------------------------------------------------
[10/10] Starting Fold 9 (100.0%)


[10/10] Finished Fold 9
   Metrics: {'accuracy': 0.7067771597230106, 'precision': 0.7131569277383999, 'recall': 0.7077496520932086, 'f1': 0.7091284079915484, 'fold': 9}
   Fold time: 65.87 secs
   Elapsed:   508.15 secs
--------------------------------------------------
All folds completed.
Total time: 508.23 seconds


,accuracy,precision,recall,f1,fold
0,0.720059,0.727602,0.719505,0.721973,0
1,0.731442,0.738526,0.731755,0.734428,1
2,0.717480,0.742331,0.717838,0.723408,2
3,0.721308,0.728475,0.721119,0.723339,3
4,0.739357,0.742372,0.738694,0.740133,4
5,0.722216,0.726828,0.721018,0.722645,5
6,0.720173,0.722854,0.721086,0.721607,6
7,0.718470,0.722211,0.718659,0.719692,7
8,0.727665,0.735405,0.727577,0.730556,8
9,0.706777,0.713157,0.707750,0.709128,9


In [9]:
# ======================================================
# CELL 10: SAVE FINAL RESULTS
# ======================================================

import os

avg = results_df[["accuracy", "precision", "recall", "f1"]].mean()

print("Average Metrics:")
print(avg)

BASE_DIR = "/app"
metrics_dir = os.path.join(BASE_DIR, "outputs", "metrics")
os.makedirs(metrics_dir, exist_ok=True)

out_path = os.path.join(metrics_dir, "boosting_results.csv")
results_df.to_csv(out_path, index=False)

print("Saved:", out_path)

Average Metrics:
accuracy     0.722495
precision    0.729976
recall       0.722500
f1           0.724691
dtype: float64
Saved: /app/outputs/metrics/boosting_results.csv


In [ ]:
#confusion matrix
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

pdf = preds_df.select("label", "prediction").toPandas()

y_true = pdf["label"].astype(int)
y_pred = pdf["prediction"].astype(int)

cm = confusion_matrix(y_true, y_pred)

disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot(values_format="d")

plt.title("Confusion Matrix")
plt.tight_layout()
plt.show()


In [ ]:
#Metrics Comparison Bar Chart
import pandas as pd
import matplotlib.pyplot as plt

boost_avg = boost.mean(numeric_only=True)
log_avg = log.mean(numeric_only=True)
tree_avg = tree.mean(numeric_only=True)
bag_avg = bag.mean(numeric_only=True)

metrics = ["accuracy", "precision", "recall", "f1"]

data = pd.DataFrame({
    "Boosting": boost_avg[metrics],
    "Bagging": bag_avg[metrics],
    "Decision Tree": tree_avg[metrics],
    "Logistic": log_avg[metrics]
}, index=metrics)

ax = data.plot(kind="bar", figsize=(10, 6))

plt.ylim(0, 1)
plt.title("Model Performance Comparison")
plt.ylabel("Score")
plt.grid(axis="y")
plt.tight_layout()
plt.show()

In [ ]:
#F1 Score Across Folds
import matplotlib.pyplot as plt

boost_s = boost.sort_values("fold")
bag_s = bag.sort_values("fold")
tree_s = tree.sort_values("fold")
log_s = log.sort_values("fold")

plt.figure(figsize=(10, 6))

plt.plot(boost_s["fold"], boost_s["f1"], marker="o", label="Boosting")
plt.plot(bag_s["fold"], bag_s["f1"], marker="o", label="Bagging")
plt.plot(tree_s["fold"], tree_s["f1"], marker="o", label="Decision Tree")
plt.plot(log_s["fold"], log_s["f1"], marker="o", label="Logistic")

plt.xlabel("Fold")
plt.ylabel("F1 Score")
plt.title("F1 Score Across Cross-Validation Folds")

plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
#Heatmap Confusion Matrix
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix

pdf = preds_df.select("label", "prediction").toPandas()

y_true = pdf["label"].astype(int)
y_pred = pdf["prediction"].astype(int)

cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(7, 6))
plt.imshow(cm, interpolation="nearest")
plt.title("Confusion Matrix Heatmap")
plt.xlabel("Predicted")
plt.ylabel("Actual")


for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        plt.text(j, i, str(cm[i, j]), ha="center", va="center")

plt.colorbar()
plt.tight_layout()
plt.show()